# Placeholder_v0.1: first model in the Placeholder series

**Station:** model training, in the assembly line of cleaning, feature engineering, enhancement, training and evaluation.
**Plan:** `docs/plans/placeholder-training-plan.md` (reviewed by Grok and Cursor). **Model code:** `notebooks/Meet/placeholder_v0_1.py`.

The model forecasts each metric as the official 30-year city mean plus a learned, shrunk shift:

`forecast = ref30(city, origin) + lambda_metric * shift_hat(features, lead)`

- **Why a shift.** The score compares us with that same mean, so the model only has to learn how far a city has moved from it.
- **Why pooled and direct.** A single city's recent trend is noisy, and feeding predictions back in (the starter random forest) compounds noise. So one model per metric learns from every city and lead at once, with every feature frozen at the forecast origin.
- **Why shrink.** One `lambda` per metric is chosen on earlier inner origins. If nothing beats the reference, `lambda = 0` returns exactly the 30-year mean.

**Rules.** Development origins 1983, 1990 and 1997 of suite-v3 only. No origin-2004/2011 data, no `private/` folder, no weather after 2018. `submission.csv` is not touched, and no origin-2018 forecast is made here.

In [1]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "code"), str(ROOT / "notebooks/Meet")]

import placeholder_v0_1 as model
from common import METRICS

NAME = model.MODEL_NAME
ADAPTER = "notebooks/Meet/placeholder_v0_1.py:forecast"
ASSETS = "notebooks/Meet/placeholder-v0.1-assets.json"
SUITE = ROOT / "outputs/evaluation/suite-v3"
WORK = ROOT / "outputs/placeholder/v0.1"
WORK.mkdir(parents=True, exist_ok=True)
DEV_ORIGINS = (1983, 1990, 1997)
pd.set_option("display.width", 160, "display.max_columns", 40)

def run(*command):
    result = subprocess.run([sys.executable, *map(str, command)], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout[-6000:])
    if result.returncode:
        print(result.stderr[-4000:])
        raise RuntimeError(f"command failed: {command}")
    return result.stdout

def completed_run(name):
    # Newest complete run of this name, same suite and same adapter bytes, so re-running the notebook adds no trial.
    from evaluation.core import digest
    from evaluation.protocol import PROTOCOL_VERSION

    suite_id = json.loads((SUITE / "manifest.json").read_text(encoding="utf-8"))["suite_id"]
    adapter_sha = digest(ROOT / ADAPTER.rsplit(":", 1)[0])
    for path in sorted((ROOT / "outputs/evaluation/runs").glob("*/run.json"), reverse=True):
        meta = json.loads(path.read_text(encoding="utf-8"))
        if (meta.get("model"), meta.get("status"), meta.get("suite_id"), meta.get("adapter_sha256")) == (name, "complete", suite_id, adapter_sha) and meta.get("protocol_version") == PROTOCOL_VERSION:
            return path.parent
    return None

print(NAME, "| root:", ROOT)

Placeholder_v0.1 | root: <repo>


## 1. What the model sees

Here is the feature table on the earliest development origin, using only that fold's training history. Every number uses years up to the pseudo-origin `t`, and regional features use the 15 nearest other cities.

In [2]:
origin = 1983
train = pd.read_csv(SUITE / f"origin-{origin}/train.csv", dtype={"city_id": str})
cities = sorted(train.city_id.unique())
coords = pd.read_csv(ROOT / "data/city_info.csv", dtype={"city_id": str}, usecols=["city_id", "lat", "lon"]).set_index("city_id")
neighbours = model.neighbour_index(model._check_coords(coords, cities))
tables = model.tables_at(train, cities, origin)
rows = model.training_rows(tables, neighbours, origin, "all")
names = model.feature_names("all")

summary = pd.DataFrame({m: {"training rows": len(y), "median shift": np.median(y), "shift IQR": np.subtract(*np.percentile(y, [75, 25]))}
                        for m, (X, y) in rows.items()}).T
print("Features:", names)
display(summary.round(3))
X, y = rows["annual_avg_temp"]
display(pd.DataFrame(X[:4], columns=names).round(2))

Features: ['gap_lag1', 'gap5', 'gap10', 'gap15', 'slope30', 'trend_gap', 'ref30', 'sigma30', 'n_valid30', 'reg_gap10', 'reg_slope30', 'nat_gap10', 'nat_slope30', 'tavg_gap10', 'tavg_trend_gap', 'h']


,training rows,median shift,shift IQR
annual_avg_temp,56404.0,-0.366,1.767
total_precip_in,56650.0,0.258,8.374
hottest_day_temp,56404.0,-0.727,4.267
cdd,56404.0,-30.450,240.167


,gap_lag1,gap5,gap10,gap15,slope30,trend_gap,ref30,sigma30,n_valid30,reg_gap10,reg_slope30,nat_gap10,nat_slope30,tavg_gap10,tavg_trend_gap,h
0,0.82,1.18,0.83,0.55,0.07,0.98,61.60,0.95,21.0,0.53,0.06,0.61,0.05,0.83,0.98,1.0
1,0.75,0.04,0.57,0.14,0.05,0.64,52.60,1.26,25.0,0.54,0.03,0.61,0.05,0.57,0.64,1.0
2,1.17,1.04,0.49,0.60,0.07,1.07,53.96,1.16,30.0,0.40,0.05,0.61,0.05,0.49,1.07,1.0
3,0.90,0.09,0.36,0.19,0.04,0.54,65.19,1.12,25.0,0.60,0.05,0.61,0.05,0.36,0.54,1.0


## 2. Unit tests

The tests run on synthetic data only. They cover the information boundaries (nothing after a pseudo-origin or inner origin can change its features or fits), `lambda = 0` equalling the reference on every row, the selection rule, determinism and neighbour handling.

In [3]:
run("-m", "unittest", "notebooks/Meet/test_placeholder_v0_1.py")
print("All Placeholder tests passed.")


All Placeholder tests passed.


## 3. Evaluate through the team's standard loop

`scripts/eval_model.py` verifies the suite, builds or reuses the control panel, runs the worker on the development origins with the official scorer and runtime audit, writes the scorecard, prints the gates and mirrors to MLflow.

In [4]:
print(run("code/evaluate.py", "verify", "--suite", SUITE))
run_dir = completed_run(NAME)
if run_dir is None:
    output = run("scripts/eval_model.py", "--name", NAME, "--adapter", ADAPTER, "--assets", ASSETS, "--suite", SUITE, "--profile", "full", "--timeout", 900)
    run_dir = Path(next(line[5:].strip() for line in output.splitlines() if line.startswith("Run: ")))
else:
    print("Reusing the sealed run; no new trial.")
scorecard_path = ROOT / "outputs/evaluation/scorecards" / f"{run_dir.name}.json"
print("Run dir:", run_dir)

Verified suite: 4029caa1fb85cd376baf677e5d89549655684044aac68b6b3efb368eb6e46872

Verified suite: 4029caa1fb85cd376baf677e5d89549655684044aac68b6b3efb368eb6e46872

Reusing the sealed run; no new trial.
Run dir: <repo>\outputs\evaluation\runs\20261108T111421Z-b7a95baa04


## 4. Pre-declared ablations

These four runs measure what each part contributes. They are fixed in the plan, use the same folds and seed, and cannot replace v0.1. Each changes exactly one thing: the learner, the gate, or the pooling.

They run only when `PLACEHOLDER_RUN_ABLATIONS=1` is set, because they need the shared evaluator source to be held stable. Finished ablations are reused, so re-running adds no trial.

In [5]:
ABLATIONS = {
    "lin": {"members": "lin"},
    "gbm": {"members": "gbm"},
    "lambda1": {"gate": "none"},
    "no_pool": {"features": "no_pool"},
}
RUN_NEW = os.environ.get("PLACEHOLDER_RUN_ABLATIONS") == "1"
ablation_runs = {}
for label, params in ABLATIONS.items():
    name = f"{NAME}-ablation-{label}"
    found = completed_run(name)
    if found is None and RUN_NEW:
        path = WORK / f"ablation-{label}.json"
        path.write_text(json.dumps(params), encoding="utf-8")
        out = run("scripts/eval_model.py", "--name", name, "--adapter", ADAPTER, "--assets", ASSETS,
                  "--params", path, "--suite", SUITE, "--profile", "full", "--timeout", 900, "--no-report")
        found = Path(next(line[5:].strip() for line in out.splitlines() if line.startswith("Run: ")))
    if found is None:
        print(f"{label}: pending (needs a stable evaluator checkpoint)")
    else:
        ablation_runs[label] = found
        print(f"{label}: {found.name}")

lin: 20261108T112644Z-9d238d0158
gbm: 20261108T112816Z-af449a65cf
lambda1: 20261108T112939Z-2d56e97ca7
no_pool: 20261108T113035Z-96f303a19c


## 5. Results against the pre-registered gates

The gates are the team's screening heuristics, as printed by `scripts/eval_model.py`. With three reused folds, a difference smaller than the spread across origins is noise.

**Expected null result.** If the gate picks `lambda = 0` for every metric and origin, v0.1 equals the 30-year average: headline 0, so G1 fails; G2 passes, because the best control excludes the average and is negative on these folds; G3 passes; G4 fails, because combo is positive on 1997. That would be a valid result for a first baseline, not a harness failure.

In [6]:
card = json.loads(scorecard_path.read_text(encoding="utf-8"))
comparisons = json.loads(scorecard_path.with_name(scorecard_path.stem + ".comparisons.json").read_text(encoding="utf-8"))

gate_table = pd.DataFrame(comparisons["gates"]).assign(**{"pass": lambda d: d["pass"].map({True: "PASS", False: "FAIL"})})
display(gate_table)

def card_of(path):
    return json.loads((ROOT / "outputs/evaluation/scorecards" / f"{path.name}.json").read_text(encoding="utf-8"))

summary_rows = []
for label, path in {NAME: run_dir, **{f"ablation:{k}": v for k, v in ablation_runs.items()}}.items():
    c = card_of(path)
    summary_rows.append({"model": label, "headline": c["fidelity"]["headline_final_raw_skill"],
                         "score_final": c["fidelity"]["official_score_final_foldmean"],
                         "score_live": c["fidelity"]["official_score_live_foldmean"],
                         **{m: c["per_metric"]["final"][m] for m in METRICS}})
results = pd.DataFrame(summary_rows).set_index("model")
results.to_csv(WORK / "results.csv")
display(results.round(4))

,gate,pass,detail
0,G1 beats the 30-year average,FAIL,headline -0.0008
1,G2 beats the best control,PASS,mean10 -0.0087
2,G3 no metric below -0.02,PASS,worst annual_avg_temp -0.0042
3,"G4 beats combo (mean, >=2 origins, not losing ...",FAIL,origin deltas 1983:+0.0301 1990:+0.0337 1997:-...


,headline,score_final,score_live,annual_avg_temp,total_precip_in,hottest_day_temp,cdd
model,,,,,,,
Placeholder_v0.1,-0.0008,0.1678,0.2475,-0.0042,0.0000,-0.0034,0.0045
ablation:lin,-0.0041,0.0000,0.0000,-0.0062,0.0000,-0.0066,-0.0038
ablation:gbm,0.0047,0.4673,0.4370,0.0033,0.0000,0.0014,0.0139
ablation:lambda1,-0.0304,0.9907,0.9965,-0.0764,-0.0319,-0.0225,0.0091
ablation:no_pool,-0.0045,0.1809,0.1765,-0.0155,0.0000,-0.0094,0.0068


In [7]:
lambdas = pd.DataFrame({o: json.loads((run_dir / f"origin-{o}/placeholder_selection.json").read_text(encoding="utf-8"))["selected"]
                        for o in DEV_ORIGINS}).T.rename_axis("origin")
lambdas.to_csv(WORK / "selected_lambdas.csv")
print("Selected shrinkage per origin and metric:")
display(lambdas)

inner_rows = []
for o in DEV_ORIGINS:
    selection = json.loads((run_dir / f"origin-{o}/placeholder_selection.json").read_text(encoding="utf-8"))
    for fold in selection["inner_folds"]:
        for lam, skills in fold["final_skill"].items():
            for metric, skill in skills.items():
                inner_rows.append({"outer_origin": o, "inner_origin": fold["origin"], "lambda": float(lam), "metric": metric, "final_skill": skill})
inner_skill = pd.DataFrame(inner_rows)
inner_skill.to_csv(WORK / "inner_fold_skill.csv", index=False)
print("Mean inner-fold final skill, every lambda and metric (nine folds; origins repeat across the three outer folds):")
display(inner_skill.pivot_table(index="metric", columns="lambda", values="final_skill").round(4))

by_origin = pd.Series(card["stability"]["final_raw_skill_by_origin"], name="final raw skill").rename_axis("origin")
print("Final raw skill by origin:")
display(by_origin.round(4).to_frame())

print(f"Versus combo:         mean delta {comparisons['vs_combo']['mean_final_raw_skill_delta_second_minus_first']:+.4f}")
print(f"Versus best control ({comparisons['best_control']}): mean delta {comparisons['vs_best_control']['mean_final_raw_skill_delta_second_minus_first']:+.4f}")
print("Per-origin delta vs combo:", {p["origin"]: round(p["raw_skill_delta"], 4) for p in comparisons["vs_combo"]["periods"]})

Selected shrinkage per origin and metric:


,annual_avg_temp,total_precip_in,hottest_day_temp,cdd
origin,,,,
1983,0.5,0.0,0.50,0.50
1990,0.0,0.0,0.25,0.25
1997,0.0,0.0,0.00,0.00


Mean inner-fold final skill, every lambda and metric (nine folds; origins repeat across the three outer folds):


lambda,0.00,0.25,0.50,0.75,1.00
metric,,,,,
annual_avg_temp,0.0,0.0072,-0.0071,-0.0417,-0.0946
cdd,0.0,0.0107,0.0096,-0.0025,-0.0242
hottest_day_temp,0.0,0.0098,0.0112,0.0045,-0.0097
total_precip_in,0.0,-0.0035,-0.0081,-0.0140,-0.0208


Final raw skill by origin:


,final raw skill
origin,
1983,-0.0073
1990,0.0050
1997,0.0000


Versus combo:         mean delta +0.0127
Versus best control (mean10): mean delta +0.0080
Per-origin delta vs combo: {1983: 0.0301, 1990: 0.0337, 1997: -0.0257}


## 6. MLflow

`scripts/eval_model.py` already mirrored each verified run through `code/evaluation/tracking.py`, the single writer. This cell only adds to those same records: series tags, model settings, selected shrinkage and fold artifacts. It creates no new run and writes nothing into the sealed run directories.

In [8]:
import mlflow
from mlflow.tracking import MlflowClient
from evaluation.tracking import experiment_name, tracking_uri

uri = tracking_uri()
mlflow.set_tracking_uri(uri)
client = MlflowClient(tracking_uri=uri)

def add_to_mlflow(path, ablation=None):
    meta = json.loads((path / "run.json").read_text(encoding="utf-8"))
    experiment = client.get_experiment_by_name(experiment_name(meta))
    hits = [] if experiment is None else client.search_runs([experiment.experiment_id], filter_string=f"tags.`harness.run_id` = '{meta['run_id']}'")
    if len(hits) != 1:
        raise RuntimeError(f"expected one MLflow run for {meta['run_id']}, found {len(hits)}; run scripts/mlflow_sync.py")
    rid = hits[0].info.run_id
    client.set_tag(rid, "series", "Placeholder")
    client.set_tag(rid, "version", "v0.1")
    client.set_tag(rid, "placeholder.ablation", ablation or "none")
    record = json.loads((path / "origin-1997/placeholder_selection.json").read_text(encoding="utf-8"))
    for key in ("members", "gate", "features", "train_years", "n_neighbours"):
        client.log_param(rid, f"placeholder.{key}", str(record[key]))
    client.log_param(rid, "placeholder.feature_names", ",".join(record["feature_names"]))
    client.log_param(rid, "placeholder.huber", json.dumps(record["huber"]))
    client.log_param(rid, "placeholder.gbm", json.dumps(record["gbm"]))
    for origin in DEV_ORIGINS:
        folder = path / f"origin-{origin}"
        selection = json.loads((folder / "placeholder_selection.json").read_text(encoding="utf-8"))
        for metric, value in selection["selected"].items():
            client.log_metric(rid, f"placeholder.lambda.{origin}.{metric}", float(value))
        client.log_artifact(rid, str(folder / "placeholder_selection.json"), artifact_path=f"origin-{origin}")
        client.log_artifact(rid, str(folder / "placeholder_models.joblib"), artifact_path=f"origin-{origin}")
    return rid

mlflow_id = add_to_mlflow(run_dir)
client.log_artifact(mlflow_id, str(scorecard_path.with_name(scorecard_path.stem + ".comparisons.json")))
client.log_artifact(mlflow_id, str(WORK / "results.csv"))
client.log_artifact(mlflow_id, str(WORK / "selected_lambdas.csv"))
client.log_artifact(mlflow_id, str(WORK / "inner_fold_skill.csv"))
for label, path in ablation_runs.items():
    add_to_mlflow(path, ablation=label)
print("MLflow run:", mlflow_id, "| experiment:", experiment_name(json.loads((run_dir / "run.json").read_text(encoding="utf-8"))))
print("Browse with: uv run mlflow ui --backend-store-uri", uri)

<repo>\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/08 06:33:43 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at <repo>\.venv\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow run: 265bdfef7e8e40109206922222b6acae | experiment: protostar/4029caa1fb85cd376baf677e5d89549655684044aac68b6b3efb368eb6e46872/development/parent-audited-v6/official
Browse with: uv run mlflow ui --backend-store-uri sqlite:///<repo>/outputs/mlflow/mlflow.db


## 7. Reading the result

The model card in `docs/models/placeholder.md` holds the conclusion and the next version's hypothesis. This notebook does not change `submission.csv`, and v0.1 does not produce a 2019-2025 forecast: that runs only after a team-approved freeze and one-shot assessment.